In [ ]:
from pathlib import Path
import json
from datetime import datetime
import pandas as pd


def find_results_dir() -> Path:
    """Encontra artifacts/results subindo a árvore de diretórios a partir do cwd."""
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / "artifacts" / "results"
        if candidate.exists() and candidate.is_dir():
            return candidate
    raise FileNotFoundError("Não foi possível localizar o diretório artifacts/results a partir do diretório atual.")

def find_score_file_for_folder(folder: Path):
    """
    Procura arquivos scores*.json dentro da pasta.
    Se houver mais de um, retorna o mais recentemente modificado.
    """
    matches = sorted(folder.rglob("scores*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0] if matches else None

import json
from datetime import datetime
from pathlib import Path

def load_score_row(run_folder: Path, score_file: Path, results_dir: Path) -> dict:
    row = {
        "run_folder": run_folder.name,
        "score_file": str(score_file.relative_to(results_dir)),
        "modified_at": datetime.fromtimestamp(score_file.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
    }

    text = score_file.read_text(encoding="utf-8").strip()
    if not text:
        row["json_error"] = "empty file"
        return row

    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        row["json_error"] = f"{e.msg} (line {e.lineno}, col {e.colno})"
        row["raw_json"] = text[:500]
        return row

    # Expande chaves do JSON para colunas
    if isinstance(data, dict):
        row.update(data)
    else:
        row["raw_json"] = str(data)

    return row


results_dir = find_results_dir()
run_folders = sorted([
    p for p in results_dir.iterdir()
    if p.is_dir() and p.name != "compare_results"
])

rows = []
missing = []

for folder in run_folders:
    score_file = find_score_file_for_folder(folder)
    if score_file is None:
        missing.append(folder.name)
        continue
    rows.append(load_score_row(folder, score_file, results_dir))

df = pd.DataFrame(rows)

if not df.empty:
    # Ordenação preferencial por score primário e secundário
    sort_cols = [c for c in ["score", "score_secondary"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(by=sort_cols, ascending=False).reset_index(drop=True)

    # Colunas de identificação primeiro
    lead_cols = [c for c in ["run_folder", "score_file", "modified_at"] if c in df.columns]
    other_cols = [c for c in df.columns if c not in lead_cols]
    df = df[lead_cols + other_cols]

print(f"Diretório analisado: {results_dir}")
print(f"Pastas com score encontrado: {len(rows)}")
print(f"Pastas sem score: {len(missing)}")
if missing:
    print("Sem scores*.json:", ", ".join(missing))

display(df)

# Opcional: salvar tabela consolidada
output_csv = results_dir / "compare_results" / "scores_comparativo.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"Tabela salva em: {output_csv}")

Diretório analisado: /home/ia368/projetos/imageclef2026-rag/artifacts/results
Pastas com score encontrado: 4
Pastas sem score: 1
Sem scores*.json: rag_medgemma_27b_20260318_213004


,run_folder,score_file,modified_at,score,score_secondary,bert,rouge,similarity,bleurt,medcat,align
0,rag_base_20260223_204535,rag_base_20260223_204535/scores_20260223_20453...,2026-03-09 15:59:11,0.499181,0.154430,0.594663,0.221224,0.874318,0.306520,0.149102,0.159757
1,rag_few_shot_20260312_103549,rag_few_shot_20260312_103549/scores.json,2026-03-22 18:40:49,0.478106,0.131079,0.581801,0.205245,0.835839,0.289540,0.112642,0.149517
2,rag_simple_prompt_20260310_113904,rag_simple_prompt_20260310_113904/scores.json,2026-03-22 18:51:23,0.457017,0.149508,0.568059,0.183479,0.813368,0.263162,0.102806,0.196209
3,baseline,baseline/scores.json,2026-03-22 19:00:01,0.342700,0.000000,0.608600,0.271000,0.932400,0.325700,0.172200,0.129900


Tabela salva em: /home/ia368/projetos/imageclef2026-rag/artifacts/results/compare_results/scores_comparativo.csv
